# Multiple environments with GroupEnv

Pass a `list[EnvConfig]` to `make_group_env` to combine several environment instances into a `GroupEnv`. Each config creates one independent `SingleEnv`. All env instances live in one flat list — `step()` takes and returns one entry per env instance.

This notebook runs **CartPole-v1** (2 env instances) and **MountainCar-v0** (3 env instances) together, giving 5 env instances total. It prints their specs and runs a short rollout loop.

## When to use multiple configs

The multi-env design is useful when your training loop needs to alternate between environments that have genuinely different observation and action spaces — for example, a curriculum that trains on easy envs and hard envs together, or a multi-task setup where each task is a different Gymnasium env.

Each config is fully independent: different `id`, `reset_seed`, constructor kwargs, and reset options. `GroupEnv` steps all env instances sequentially and returns a single flat `list[dict]` of outputs — one entry per env instance. Episode statistics accumulate in `env.tracker` (`GroupTracker`) and can be cleared with `env.tracker.clear()`.

## `env.names` and env indexing

`env.names` is the full flat list of env instance names across all configs:

```
("CartPole-v1_0", "CartPole-v1_1", "MountainCar-v0_0", "MountainCar-v0_1", "MountainCar-v0_2")
```

`env.num_envs` is the total env instance count (5 here). `outputs[0]` and `outputs[1]` are CartPole env instances; `outputs[2]`, `outputs[3]`, `outputs[4]` are MountainCar env instances.

## Imports

In [ ]:
from mouse_gym import EnvConfig, make_env, make_group_env

## Build a multi-environment

`make_env` accepts a plain list of configs. Each entry is independent — different ids, seeds, names, action spaces, and observation spaces are all fine.

In [ ]:
env = make_group_env([
    EnvConfig(id="CartPole-v1", reset_seed=0, name="CartPole-v1_0", episodes_per_task=100),
    EnvConfig(id="CartPole-v1", reset_seed=1, name="CartPole-v1_1", episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", reset_seed=2, name="MountainCar-v0_0", episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", reset_seed=3, name="MountainCar-v0_1", episodes_per_task=100),
    EnvConfig(id="MountainCar-v0", reset_seed=4, name="MountainCar-v0_2", episodes_per_task=100),
])

## Inspect specs

`env.output_specs` and `env.input_specs` are flat lists — one entry per env instance (5 total). `env.names` is the full flat list of env instance names.

In [ ]:
print(f"Total env instances : {env.num_envs}")
print(f"All names           : {env.names}")
print()
for name, ospec, ispec in zip(env.names, env.output_specs, env.input_specs):
    print(f"Env {name}")
    print(f"  obs  dtype={ospec.observation.dtype}  shape={ospec.observation.shape}")
    print(f"  act  dtype={ispec.action.dtype}        shape={ispec.action.shape}")

## Rollout loop

`env.sample_random_input()` returns `list[dict]` — one dict per env instance (5 total).
`env.step(inputs)` returns a flat `list[dict]` of outputs — one entry per env instance.

Episode statistics accumulate in `env.tracker`. Call `env.tracker.clear()` to reset
between evaluation runs. Slice by index range to separate CartPole from MountainCar results.

In [ ]:
for step in range(300):
    env.step(env.sample_random_input())

# Episode stats accumulated in env.tracker automatically during the rollout
for env_index, rewards in enumerate(env.tracker.episode_cum_rewards):
    name = env.names[env_index]
    avg = sum(rewards) / len(rewards) if rewards else float("nan")
    print(f"{name:25s}  episodes={len(rewards):3d}  avg_return={avg:.2f}")

env.close()